## NVDA Time Series – Main Notebook

This notebook replaces the previous `main.py` script as the primary interactive entry point for the NVDA projects.

Use it to:
- Download and load the NVDA dataset.
- Run preprocessing and basic data description.
- Inspect the DataFrame with commands like `df.head()`.
- Later: perform transformations to returns and add visualizations (after we verify the time-series structure).


In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from statsmodels.tsa.stattools import adfuller


from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from scipy.stats import randint, uniform

from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler

# Data import and cleaning

In [ ]:
df = yf.download(
    tickers="NVDA",
    start="2021-01-01",
    end="2026-03-05",
    auto_adjust=True,
)

if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.get_level_values(0)

df.index = pd.to_datetime(df.index)
print(df.shape)
df.head()

<div style="background: #d8dbe0; padding: 10px; border-radius: 6px; color: rgb(0, 0, 0); width: 98%">
    <details>
        <summary><strong>Section 1 — Data Download</strong></summary>
        <ul>
            <li>NVIDIA stock data (<code>NVDA</code>) was downloaded using the <code>yfinance</code> library covering the period <strong>January 2021 – March 2026</strong>, providing approximately five years of daily trading data.</li>
            <li>The <code>auto_adjust=True</code> parameter ensures prices are adjusted for stock splits and dividends — important for NVIDIA given its <strong>10-for-1 stock split in June 2024</strong>, which would otherwise create an artificial price discontinuity.</li>
            <li>The date range was chosen to capture both the pre-AI-boom period (2021–2022) and the high-growth AI-driven period (2023–2026), providing structural diversity in the data.</li>
        </ul>
    </details>
</div>

In [ ]:
print("Shape:", df.shape)
print("\nDtypes:\n", df.dtypes)
print("\nMissing values:\n", df.isnull().sum())
print("\nDate range:", df.index.min(), "→", df.index.max())
df.describe()

<div style="background: #d8dbe0; padding: 10px; border-radius: 6px; color: rgb(0, 0, 0); width: 98%">
    <details>
        <summary><strong>Section 2 — Data Inspection</strong></summary>
        <ul>
            <li>Raw data was inspected for shape, data types, and missing values before any modifications — standard practice to understand what cleaning steps are actually necessary rather than applying them blindly.</li>
            <li>The dataset contains five OHLCV columns: <code>Open</code>, <code>High</code>, <code>Low</code>, <code>Close</code>, <code>Volume</code>. Since <code>auto_adjust=True</code> was used, prices are already adjusted and no separate <code>Adj Close</code> column appears.</li>
            <li>The index is a <code>DatetimeIndex</code> at daily frequency, covering only trading days — weekends and public holidays are naturally absent, which is expected behavior for financial data.</li>
        </ul>
    </details>
</div>

In [ ]:
df = df[~df.index.duplicated(keep="first")]

df = df.dropna(subset=["Close"])

df = df.ffill()

print("Shape after cleaning:", df.shape)
print("Missing values:\n", df.isnull().sum())

<div style="background: #d8dbe0; padding: 10px; border-radius: 6px; color: rgb(0, 0, 0); width: 98%">
    <details>
        <summary><strong>Section 3 — Data Cleaning</strong></summary>
        <ul>
            <li>Duplicate index rows were removed using <code>df.index.duplicated(keep="first")</code> — rare in yfinance data but possible if API calls overlap on boundary dates.</li>
            <li>Rows where <code>Close</code> is missing were dropped first, since <code>Close</code> is the foundation for all derived features and the target variable — any row without it is unusable.</li>
            <li>Remaining gaps were forward-filled using <code>ffill()</code>, which carries the last known value forward. This is preferred over mean imputation for time series because it respects temporal order and does not introduce future information.</li>
            <li>After cleaning, missing value counts were confirmed to be zero across all columns before proceeding to feature engineering.</li>
        </ul>
    </details>
</div>

# Feature Engineering

In [ ]:
df["Log_Return"] = np.log(df["Close"] / df["Close"].shift(1))

In [ ]:
df["Lag_1"] = df["Log_Return"].shift(1)
df["Lag_2"] = df["Log_Return"].shift(2)
df["Lag_5"] = df["Log_Return"].shift(5)

In [ ]:
df["Volatility_10"] = df["Log_Return"].rolling(window=10).std()
df["Volatility_20"] = df["Log_Return"].rolling(window=20).std()

In [ ]:
rolling_mean = df["Close"].rolling(window=20).mean()
rolling_std  = df["Close"].rolling(window=20).std()
bb_upper     = rolling_mean + (2 * rolling_std)
bb_lower     = rolling_mean - (2 * rolling_std)
bb_width     = bb_upper - bb_lower

df["BB_Pct"] = (df["Close"] - bb_lower) / bb_width

In [ ]:
df["Volume_Change"] = df["Volume"].pct_change()

In [ ]:
sp500 = yf.download("^GSPC", start="2021-01-01", end="2026-03-05", auto_adjust=True)["Close"].squeeze()
vix   = yf.download("^VIX",  start="2021-01-01", end="2026-03-05", auto_adjust=True)["Close"].squeeze()

df["SP500_Return"] = np.log(sp500 / sp500.shift(1))
df["VIX"] = vix

In [ ]:
df["Target"] = df["Log_Return"].shift(-1)

In [ ]:
df = df.dropna()

print("Final shape:", df.shape)
print("Missing values:\n", df.isnull().sum())

<div style="background: #d8dbe0; padding: 10px; border-radius: 6px; color: rgb(0, 0, 0); width: 98%">
    <details>
        <summary><strong>Section 4 — Feature Engineering</strong></summary>
        <ul>
            <li>Raw OHLCV data alone provides insufficient regressors for a meaningful regression task — feature engineering expands the input space from 5 raw columns to <strong>10 economically motivated features</strong>.</li>
            <li><strong>Log returns</strong> (<code>Log_Return</code>) are computed as <code>ln(Close_t / Close_{t-1})</code> rather than simple percentage returns — they are approximately normally distributed, time-additive, and stationary, making them the standard choice in financial modelling.</li>
            <li><strong>Lagged returns</strong> (<code>Lag_1</code>, <code>Lag_2</code>, <code>Lag_5</code>) frame the problem as a supervised learning task — past returns become predictors of the next day's return, consistent with autoregressive modelling logic.</li>
            <li><strong>Rolling volatility</strong> (<code>Volatility_10</code>, <code>Volatility_20</code>) captures short-term and medium-term volatility regimes respectively. Both were retained despite correlation, as they represent distinct time horizons and provide interpretable contrast in the coefficient plots.</li>
            <li><strong>Bollinger Band %</strong> (<code>BB_Pct</code>) measures where the current price sits within its 20-day band — bounded approximately between 0 and 1, stationary, and more informative than raw moving averages which are non-stationary.</li>
            <li><strong>Volume Change</strong> captures relative day-over-day trading activity, which can signal institutional interest or panic selling independent of price direction.</li>
            <li><strong>External regressors</strong> (<code>SP500_Return</code>, <code>VIX</code>) introduce market-wide context. S&P 500 log returns represent systematic market risk, while VIX is a forward-looking fear gauge — conceptually distinct from the backward-looking realized volatility features already included.</li>
            <li>The <strong>target variable</strong> is defined as the next day's log return (<code>Log_Return.shift(-1)</code>), ensuring no same-day information is used to predict same-day outcomes — a common source of feature leakage in stock prediction tasks that was explicitly avoided here.</li>
        </ul>
    </details>
</div>

# EDA

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))

ax.plot(df.index, df["Close"], color="#76b900", linewidth=1.5)  

ax.set_title("NVIDIA (NVDA) Closing Price — 2021 to 2026", fontsize=14)
ax.set_xlabel("Date")
ax.set_ylabel("Price (USD)")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12, 4))

plt.plot(df.index, df["Log_Return"], label="Log_returns", alpha=0.8)

plt.title("Log_returns over time")
plt.xlabel("Date")
plt.ylabel("Return")
plt.axhline(0.0, color="black", linewidth=0.8, linestyle="--", alpha=0.6)
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12, 4))

plt.plot(df.index, df["Volume_Change"], label="Volume_Change", alpha=0.8)

plt.title("Volume_Change over time")
plt.xlabel("Date")
plt.ylabel("Volume Change (%)")
plt.axhline(0.0, color="black", linewidth=0.8, linestyle="--", alpha=0.6)
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
df["Volume"].sort_values(ascending=False).head()

In [ ]:
plt.figure(figsize=(8, 4))
sns.histplot(data, bins=50, kde=True)

plt.xlabel("Log_Return")
plt.ylabel("Frequency")
plt.title(f"Distribution of {"Log_Return"}")
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

df_box = df[["Log_Return"]].copy()
df_box["Year"] = df.index.year

sns.boxplot(data=df_box, x="Year", y="Log_Return", ax=ax, palette="Blues")
ax.axhline(0, color="red", linewidth=0.8, linestyle="--")

ax.set_title("NVIDIA Volatility & Outliers by year", fontsize=14)
ax.set_xlabel("Year")
ax.set_ylabel("Log Return")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(df.drop(columns=["Target"]).corr(numeric_only=True), annot=True, fmt=".2f", cmap="RdBu")
plt.show()

<div style="background: #d8dbe0; padding: 10px; border-radius: 6px; color: rgb(0, 0, 0); width: 98%">
    <details>
        <summary><strong>Section 5 — Exploratory Data Analysis (EDA)</strong></summary>
        <ul>
            <li>EDA was conducted after cleaning but before modelling.</li>
            <li>The <strong>closing price line plot</strong> reveals a clear structural break around late 2022 / early 2023, coinciding with the surge in AI investment following ChatGPT's release — NVIDIA's exponential price appreciation from this point confirms that raw price levels are non-stationary and unsuitable as regression targets.</li>
            <li>The <strong>log returns line plot</strong> shows volatility clustering — periods of large swings followed by relative calm — characteristic of financial return series and the motivation behind ARCH/GARCH models in econometrics.</li>
            <li>The <strong>distribution of log returns</strong> is approximately bell-shaped and centred near zero, consistent with stationarity. However, the distribution exhibits <strong>fat tails</strong> (excess kurtosis) — extreme returns occur more frequently than a normal distribution would predict, a well-documented feature of financial data.</li>
            <li>The <strong>boxplot by year</strong> shows visibly wider interquartile ranges and more extreme outliers in 2023–2025 compared to 2021–2022, reflecting heightened volatility during the AI boom. This year-on-year variation motivates the inclusion of volatility features in the model.</li>
            <li>The <strong>correlation heatmap</strong> confirms moderate correlation between <code>Volatility_10</code> and <code>Volatility_20</code>, and between the three lag features, Ridge Regression handles multicollinearity through L2 regularization.</li>
        </ul>
    </details>
</div>

# ADF Test Staionarity

In [ ]:
def run_adf(series, name):
    result = adfuller(series.dropna(), autolag="AIC")
    stat, pval, lags, nobs, crit, _ = result

    print(f"{name}")
    print(f"ADF Statistic: {stat:.4f}")
    print(f"p-value: {pval:.4f}")
    print(f"Used lags: {lags}")
    
    for k, v in crit.items():
        print(f"Critical Value ({k}): {v:.4f}")

    if pval < 0.05:
        print("Stationary (reject H0)")
    else:
        print("Non-stationary (fail to reject H0)")
    
    print()

In [ ]:
run_adf(df["Log_Return"], "NVIDIA Log Returns")

run_adf(df["VIX"], "VIX (level series)")

run_adf(df["SP500_Return"], "S&P 500 Log Returns")

<div style="background: #d8dbe0; padding: 10px; border-radius: 6px; color: rgb(0, 0, 0); width: 98%">
    <details>
        <summary><strong>Section 6 — Stationarity Testing (ADF)</strong></summary>
        <ul>
            <li>The <strong>Augmented Dickey-Fuller (ADF) test</strong> was applied to verify stationarity before modelling — a requirement for regression on time series data, as non-stationary features can produce spurious relationships.</li>
            <li>The null hypothesis is the presence of a unit root (non-stationarity). Rejection at the 5% significance level confirms the series is stationary and safe to use as a feature or target.</li>
            <li><strong>NVDA Log Return</strong> — null hypothesis rejected (p &lt; 0.05). Daily log returns are stationary, validating the use of <code>Log_Return</code> and its lags as both features and the target variable.</li>
            <li><strong>S&P 500 Log Return</strong> — null hypothesis rejected (p &lt; 0.05). Stationary, consistent with efficient market theory that aggregate market returns are not predictable from their own history.</li>
            <li><strong>VIX (level series)</strong> — null hypothesis rejected (p &lt; 0.05). Although VIX is a level series rather than a return, its strong mean-reverting properties result in stationarity — no transformation was required.</li>
            <li>All three series passed the stationarity check, confirming the feature set is appropriate for regression modelling without further differencing.</li>
        </ul>
    </details>
</div>

# Preprocessing

In [ ]:
split_idx = int(len(df) * 0.80)

train = df.iloc[:split_idx]
test  = df.iloc[split_idx:]

features = ["Lag_1", "Lag_2", "Lag_5", "Volatility_10", "Volatility_20", "BB_Pct", "Volume_Change", "SP500_Return", "VIX"]

X_train = train[features]
y_train = train["Target"]

X_test  = test[features]
y_test  = test["Target"]

print(f"Training set: {X_train.shape[0]} observations  ({train.index[0].date()} to {train.index[-1].date()})")
print(f"Testing set: {X_test.shape[0]}  observations  ({test.index[0].date()} to {test.index[-1].date()})")

In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test) 

# Modelling

<div style="background: #d8dbe0; padding: 10px; border-radius: 6px; color: rgb(0, 0, 0); width: 98%">
    <details>
        <summary><strong>Section 7 — Model Selection & Hyperparameter Tuning</strong></summary>
        <ul>
            <li>Four models were selected to represent a natural progression of complexity: <strong>Linear Regression</strong> (acs as a baseline) → <strong>Ridge Regression</strong> (regularized linear), then <strong>Random Forest</strong> (nonlinear, tree-based), then <strong>Gradient Boosting</strong> (sequential ensemble).</li>
            <li><strong>Linear Regression</strong> has no hyperparameters and was fitted directly. <strong>Ridge</strong>'s single parameter <code>alpha</code> was selected via cross-validated MAE across candidates [0.01, 0.1, 1.0, 10, 100] — the optimal value of <strong>α=100</strong> indicates strong regularization, consistent with the moderate multicollinearity observed in the feature heatmap.</li>
            <li><strong>Random Forest and Gradient Boosting</strong> were tuned using <code>RandomizedSearchCV</code> with <code>TimeSeriesSplit(n_splits=5)</code>. Standard <code>KFold</code> was deliberately avoided — it shuffles observations randomly, which would leak future data into training folds and invalidate temporal integrity.</li>
            <li>A <strong>two-grid tuning strategy</strong> was applied: Grid 1 used wide boundaries to explore the parameter space; Grid 2 narrowed the search around Grid 1's best values with increased <code>n_iter=50</code>. A third grid was considered but omitted as improvement from Grid 1 to Grid 2 was already marginal, indicating convergence.</li>
            <li>All hyperparameter selection was performed <strong>exclusively on training data</strong>. The test set was never evaluated during tuning.</li>
            <li>Feature scaling via <code>StandardScaler</code> was applied to Linear and Ridge models only. The scaler was fitted on training data and applied to the test set, preventing distribution leakage. Tree-based models are do not require scaling and received raw features directly.</li>
        </ul>
    </details>
</div>

In [ ]:
def evaluate(name, model, X_te, y_te):
    y_pred = model.predict(X_te)
    mae = mean_absolute_error(y_te, y_pred)
    rmse = np.sqrt(mean_squared_error(y_te, y_pred))
    r2 = r2_score(y_te, y_pred)
    
    print(f"{name}")
    print(f"MAE: {mae:.6f}")
    print(f"RMSE: {rmse:.6f}")
    print(f"R²: {r2:.4f}")
    print()
    
    return {"Model": name, "MAE": round(mae, 6), "RMSE": round(rmse, 6), "R²": round(r2, 4)}

results = []

## Hyperparamter Tuning with RandomizedSeachCV

In [ ]:
SEED = 42

In [ ]:
tscv = TimeSeriesSplit(n_splits=5)


def run_randomized_search(estimator, param_grid, name, X, y, cv_strategy, n_iter=30):
    """
    Executes a RandomizedSearchCV, fits the data, and prints the best parameters and MAE.
    Returns the fitted search object.
    """
    search = RandomizedSearchCV(
        estimator=estimator,
        param_distributions=param_grid,
        n_iter=n_iter,
        scoring="neg_mean_absolute_error",
        cv=cv_strategy,
        random_state=SEED,
        n_jobs=-1,
        verbose=1
    )
    search.fit(X, y)

    params = ", ".join(f"{k}={v}" for k, v in search.best_params_.items())

    print(f"{name} — Best params: {params}")
    print(f"{name} — Best MAE: {-search.best_score_:.5f}\n")
    return search

## Baseline Models

In [ ]:
lr = LinearRegression()
lr.fit(X_train_scaled, y_train)
results.append(evaluate("Linear Regression", lr, X_test_scaled, y_test))

In [ ]:
coef_lr = pd.Series(lr.coef_, index=features).sort_values()

fig, ax = plt.subplots(figsize=(8, 5))
colors = ["#d73027" if c < 0 else "#1a9850" for c in coef_lr]
ax.barh(coef_lr.index, coef_lr.values, color=colors)
ax.axvline(0, color="black", linewidth=0.8, linestyle="--")
ax.set_title("Linear Regression — Feature Coefficients", fontsize=13)
ax.set_xlabel("Coefficient Value")
ax.grid(True, alpha=0.3, axis="x")
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.linear_model import RidgeCV 

ridge = RidgeCV(alphas=[0.01, 0.1, 1.0, 10, 100])
ridge.fit(X_train_scaled, y_train)

print(f"The best alpha found was: {ridge.alpha_}")

results.append(evaluate("Optimized Ridge", ridge, X_test_scaled, y_test))

In [ ]:
coef_ridge = pd.Series(ridge.coef_, index=features).sort_values()

fig, ax = plt.subplots(figsize=(8, 5))
colors = ["#d73027" if c < 0 else "#1a9850" for c in coef_ridge]
ax.barh(coef_ridge.index, coef_ridge.values, color=colors)
ax.axvline(0, color="black", linewidth=0.8, linestyle="--")
ax.set_title(f"Ridge Regression (α={best_alpha}) — Feature Coefficients", fontsize=13)
ax.set_xlabel("Coefficient Value")
ax.grid(True, alpha=0.3, axis="x")
plt.tight_layout()
plt.show()

In [ ]:
rf_base = RandomForestRegressor(random_state=SEED)

rf_grid_1 = {
    "n_estimators": randint(50, 500),
    "max_depth": [1, 3, 5, 10, 15, None],
    "min_samples_leaf": randint(1, 50),
}
rf_search_1 = run_randomized_search(rf_base, rf_grid_1, "RF Grid 1", X_train, y_train, tscv)

rf_grid_2 = {
    "n_estimators": randint(80, 220),
    "max_depth": [3, 4, 5],
    "min_samples_leaf": randint(7, 15),
}
best_rf = run_randomized_search(rf_base, rf_grid_2, "RF Grid 2", X_train, y_train, tscv)

In [ ]:
results.append(evaluate("Random Forest Regressor", best_rf, X_test, y_test))

In [ ]:
gb_base = GradientBoostingRegressor(random_state=SEED)

gb_grid_1 = {
    "n_estimators": randint(50, 500),
    "learning_rate": uniform(0.01, 0.3),
    "max_depth": [2, 3, 4, 5, 6],
    "min_samples_leaf": randint(1, 50),
    "subsample": uniform(0.6, 0.4), 
}
gb_search_1 = run_randomized_search(gb_base, gb_grid_1, "GB Grid 1", X_train, y_train, tscv)

gb_grid_2 = {
    "n_estimators": randint(100, 220),
    "learning_rate": uniform(0.008, 0.2),
    "max_depth": [2, 3],
    "min_samples_leaf": randint(15, 25),
    "subsample": uniform(0.68, 0.12),
}
best_gb = run_randomized_search(gb_base, gb_grid_1, "GB Grid 2", X_train, y_train, tscv)


In [ ]:
results.append(evaluate("Gradient Boosting Regressor", best_gb, X_test, y_test))

# Model Analysis and Visualization

<div style="background: #d8dbe0; padding: 10px; border-radius: 6px; color: rgb(0, 0, 0); width: 98%">
    <details>
        <summary><strong>Section 8 — Per-Model Analysis & Visualizations</strong></summary>
        <ul>
            <li><strong>Coefficient plots (Linear & Ridge)</strong> — <code>Volatility_10</code> is the strongest positive predictor while <code>Volatility_20</code> is the strongest negative, reflecting the opposing signals of short vs medium-term volatility regimes. Ridge visibly shrinks all coefficients toward zero — direct visual evidence of L2 regularization, with <code>Volatility_10</code> reducing from ~0.0033 to ~0.0022.</li>
            <li><strong>Feature importance plots (RF & GB)</strong> — both models independently rank <code>Lag_5</code> and <code>Volume_Change</code> as the most informative features, suggesting weekly momentum and volume dynamics carry more signal than the most recent day's return. The models disagree on <code>Volatility_10</code> — a known property of GB's sequential correction mechanism distributing importance more evenly across features.</li>
            <li><strong>Actual vs Predicted plots</strong> — across all four models the predicted line is nearly flat while actual returns exhibit large sharp movements. No model captures extreme return events (e.g. the ~17% single-day spike in April 2025), as these are driven by unpredictable news shocks absent from the feature set. The visual similarity across all four models is itself a key finding — complexity does not help when the signal is absent.</li>
            <li><strong>Residual plots</strong> — residuals scatter randomly around zero with no visible trend across all models, satisfying the zero mean error assumption. The heteroskedastic spread — wider residuals in early 2025, tighter mid-2025 — mirrors NVIDIA's actual volatility clustering and is a natural consequence of modelling a high-volatility asset with fixed features.</li>
        </ul>
    </details>
</div>

In [ ]:
importance_df = pd.DataFrame({
    "Random Forest": rf_search_2.best_estimator_.feature_importances_,
    "Gradient Boosting": gb_search_1.best_estimator_.feature_importances_
}, index=features).sort_values("Random Forest", ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(features))
width  = 0.35

ax.barh(x - width/2, importance_df["Random Forest"], width, label="Random Forest", color="#1f77b4")
ax.barh(x + width/2, importance_df["Gradient Boosting"], width, label="Gradient Boosting", color="#ff7f0e")

ax.set_yticks(x)
ax.set_yticklabels(importance_df.index)
ax.set_title("Feature Importances — Random Forest vs Gradient Boosting", fontsize=13)
ax.set_xlabel("Importance Score")
ax.legend()
ax.grid(True, alpha=0.3, axis="x")
plt.tight_layout()
plt.show()

In [ ]:
models = {
    "Linear Regression": (lr, X_test_scaled),
    "Ridge Regression": (ridge, X_test_scaled),
    "Random Forest": (best_rf, X_test),
    "Gradient Boosting": (best_gb, X_test),
}

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes = axes.flatten()

for i, (name, (model, X_te)) in enumerate(models.items()):
    y_pred = model.predict(X_te)
    axes[i].plot(y_test.index, y_test.values, label="Actual", color="black", linewidth=0.9, alpha=0.7)
    axes[i].plot(y_test.index, y_pred,label="Predicted", color="red", linewidth=0.9, alpha=0.7, linestyle="--")
    axes[i].set_title(name, fontsize=12)
    axes[i].set_xlabel("Date")
    axes[i].set_ylabel("Log Return")
    axes[i].legend(fontsize=9)
    axes[i].grid(True, alpha=0.3)

plt.suptitle("Actual vs Predicted Log Returns — Test Set (2025–2026)", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes = axes.flatten()

for i, (name, (model, X_te)) in enumerate(models.items()):
    residuals = y_test.values - model.predict(X_te)
    axes[i].scatter(y_test.index, residuals, color="#1f77b4", alpha=0.4, s=10)
    axes[i].axhline(0, color="red", linewidth=0.9, linestyle="--")
    axes[i].set_title(f"{name} — Residuals", fontsize=12)
    axes[i].set_xlabel("Date")
    axes[i].set_ylabel("Residual")
    axes[i].grid(True, alpha=0.3)

plt.suptitle("Residual Plots — Test Set (2025–2026)", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

# Evaliation

In [ ]:
results_df = pd.DataFrame(results).set_index("Model")
print("Comparison of Model Performance")
print(results_df.to_string())

# Discussion

<div style="background: #d8dbe0; padding: 10px; border-radius: 6px; color: rgb(0, 0, 0); width: 98%">
    <details>
        <summary><strong>Section 10 — Discussion</strong></summary>
        <ul>
            <li><strong>Stock returns are fundamentally noisy.</strong> Daily log returns are driven primarily by unpredictable news, sentiment shifts, and macroeconomic surprises — none of which are captured by technical or lagged features derived from historical prices alone.</li>
            <li><strong>Model complexity does not solve the signal problem.</strong> The near-identical flat predicted lines across all four models demonstrate that the limitation is not model capacity but the absence of exploitable structure in the target. Increasing complexity further (e.g. LSTM, XGBoost) would be unlikely to produce materially different results.</li>
            <li><strong>Results are consistent with the Efficient Market Hypothesis (EMH).</strong> Under the weak form of EMH, past prices contain no information that can reliably predict future returns — precisely what the near-zero R² values across all models reflect.</li>
            <li><strong>Look-ahead bias was avoided but real-world performance would be worse.</strong> The chronological split does not account for transaction costs, bid-ask spreads, execution latency, or market impact — all of which would further erode any marginal predictive edge in a live trading context.</li>
            <li><strong>Feature leakage was explicitly avoided.</strong> The target was defined as the <em>next day's</em> log return, ensuring no contemporaneous price information was used as a predictor — a common methodological error in regression papers on financial data.</li>
            <li><strong>Residual heteroskedasticity</strong> — wider errors during early 2025 — reflects genuine volatility clustering during the AI investment surge. Modelling this variance structure would require ARCH/GARCH models, representing a natural extension beyond the scope of this regression task.</li>
        </ul>
    </details>
</div>